<a href="https://colab.research.google.com/github/tousif47/Geostormcaster/blob/main/omni_parse_1h_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
omni_parse.py — OMNI2 hourly .dat files -> clean CSV
Expects omni2_1963.dat ... omni2_2026.dat in DATA_DIR (see README for sources).
Self-check at the end validates the parse against known solar wind physics.
"""
import pandas as pd
import numpy as np
from pathlib import Path

# --- settings ---
DATA_DIR = Path("/content/drive/MyDrive/temp")   # <- folder with the .dat files
OUT_CSV  = "/content/drive/MyDrive/temp/omni_clean.csv"

# OMNI2 hourly field order (defined by NASA's omni2.text format doc)
COLS = [
    'year', 'doy', 'hour', 'bartels',
    'imf_sc', 'sw_sc', 'n_imf', 'n_sw',
    'b_scalar', 'b_magnitude',
    'lat_angle', 'long_angle',
    'bx_gse', 'by_gse', 'bz_gse', 'by_gsm', 'bz_gsm',
    'sigma_b_scalar', 'sigma_b_vector', 'sigma_bx', 'sigma_by', 'sigma_bz',
    'temp', 'density', 'speed', 'flow_long', 'flow_lat', 'na_np',
    'flow_pressure', 'sigma_t', 'sigma_n', 'sigma_v', 'sigma_phi',
    'sigma_theta', 'sigma_nanp', 'electric_field', 'plasma_beta',
    'alfven_mach',
    'kp', 'sunspot', 'dst', 'ae',
    'pf_1mev', 'pf_2mev', 'pf_4mev', 'pf_10mev', 'pf_30mev', 'pf_60mev',
    'pf_flag', 'ap', 'f107', 'pcn', 'al', 'au', 'ms_mach', 'q_index',
    'lt', 'f107_adj',
]

# OMNI marks missing data with sentinel numbers; treat them as NaN
NA_VALUES = [9999.99, 999.99, 999.90, 99999.9, 99999.99,
             9999999.0, 999999.0, 999.999, 9999.9, 99999.0]

# --- read and stack all yearly files ---
frames = []
for path in sorted(DATA_DIR.glob("omni2_*.dat")):
    df = pd.read_csv(path, delim_whitespace=True, names=COLS,
                     na_values=NA_VALUES, low_memory=False)
    frames.append(df)
    print(f"{path.name}: {len(df)} rows")

df = pd.concat(frames, ignore_index=True)

# --- build timestamps (Year + day-of-year + hour) and sort ---
df['timestamp'] = pd.to_datetime(df['year'], format='%Y') \
    + pd.to_timedelta(df['doy'] - 1, unit='D') \
    + pd.to_timedelta(df['hour'], unit='h')
df = df.set_index('timestamp').sort_index()

# every complete year must have 8760h (8784 in leap years); last year may be partial
full_years = df[df['year'] < df['year'].max()]
assert full_years.groupby('year').size().between(8760, 8784).all(), \
    "bad row count — check .dat files"
df = df[~df.index.duplicated(keep='first')]

# --- fix Kp: stored in tenths in some releases (33 = Kp 3+); clip 9o top-code ---
if df['kp'].max() > 9:
    df['kp'] = df['kp'] / 10.0
df['kp'] = df['kp'].clip(upper=9.0)

# --- keep the columns this project needs ---
keep = ['bz_gsm', 'by_gsm', 'speed', 'density', 'temp', 'b_magnitude',
        'kp', 'f107', 'sunspot']
out = df[keep].copy()

# --- self-check: expect bz mean ~0, by mean ~0, speed ~430, kp 0..9 mean ~2 ---
# (big deviations mean a fill value leaked through — add it to NA_VALUES)
print("\n===== SELF-CHECK =====")
for c in keep:
    s = out[c].dropna()
    print(f"{c:12s} n={len(s):8d}  min={s.min():10.2f}  "
          f"max={s.max():10.2f}  mean={s.mean():10.2f}")

out.to_csv(OUT_CSV)
print(f"\nSaved {OUT_CSV}: {len(out)} rows")

/tmp/ipykernel_4768/860294249.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni2_1963.dat: 8760 rows


/tmp/ipykernel_4768/860294249.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni2_1964.dat: 8784 rows


/tmp/ipykernel_4768/860294249.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


...

omni2_2024.dat: 8784 rows


/tmp/ipykernel_4768/860294249.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni2_2025.dat: 8760 rows


/tmp/ipykernel_4768/860294249.py:39: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(path, delim_whitespace=True, names=COLS,


omni2_2026.dat: 8760 rows

===== SELF-CHECK =====
bz_gsm       n=  434622  min=    -57.80  max=     58.90  mean=     -0.02
by_gsm       n=  434622  min=    -43.10  max=     67.60  mean=     -0.05
speed        n=  430563  min=    156.00  max=   1189.00  mean=    434.14
density      n=  414242  min=      0.00  max=    137.20  mean=      6.64
temp         n=  397340  min=   3221.00  max=6632456.00  mean= 102645.09
b_magnitude  n=  434648  min=      0.10  max=     67.80  mean=      5.50
kp           n=  561024  min=      0.00  max=      9.00  mean=      2.11
f107         n=  558672  min=     63.40  max=    412.90  mean=    119.24
sunspot      n=  558816  min=      0.00  max=    428.00  mean=     84.99

Saved /content/drive/MyDrive/temp/omni_clean.csv: 561024 rows
